In [ ]:
# 01 · Easy/Medium/Hard 균등 혼합 / 기존 체크포인트 없이 처음부터
CFG = {
    # 저장 / 다른 버전과 별도 실험
    'run_name': 'moveboxes_joint_object_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes_joint',
    'project_ref': 'main',
    'output_root': '/content/moveboxes_runs',
    # T4 환경
    'repo_dir': '/content/berlin-marso-joint',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],
    # 무작위 초기화 / 모든 상자 attention / 세 난이도 공동 학습
    'seed': 42,
    'num_demos': None,
    'batch_size': 32,
    'lr': 0.0001,
    'amp': True,
    'history': 8,
    'chunk_size': 8,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'spatial_layers': 1,
    'joint_total_iters': 30000,
    'joint_eval_interval': 2000,
    'warmup_steps': 500,
    # 세 난이도 평가 / 별도 테스트
    'tuning_episodes': 8,
    'tuning_seed_start': 410000,
    'test_episodes': 8,
    'test_seed_start': 430000,
    'benchmark_episodes': 100,
    'eval_seed_start': 440000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'validation_batches': 4,
    'console_interval_seconds': 30,
    'team': 'my-team',
}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0,str(PROJECT/'hard'/'code'))
sys.path.insert(0,str(PROJECT/'unified'/'code'))
for name in ('build_unified_notebook','unified_policy','unified_data','unified_experiment'):
    if name in sys.modules: importlib.reload(sys.modules[name])
from build_joint_notebook import CONFIG as UNIFIED_DEFAULTS
CFG = dict(UNIFIED_DEFAULTS, **CFG)
for name in ('joint_data','joint_experiment','build_joint_notebook'):
    if name in sys.modules: importlib.reload(sys.modules[name])
from joint_experiment import JointExperiment, source_bundle
experiment = JointExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · 원본 세 난이도 데이터 준비 / 기존 모델 다운로드 없음
experiment.prepare()


In [ ]:
# 06 · GPU 역전파 / 같은 모델 세 난이도 로딩 / 초기화 확인
experiment.check_runtime()
print("난이도 비율: Easy ≈ Medium ≈ Hard = 1:1:1")
print("학습 예산:", experiment.cfg["joint_total_iters"], "회 / 평가 간격:", experiment.cfg["joint_eval_interval"])
_ = experiment.report()


In [ ]:
# 07 · 세 난이도 처음부터 공동 학습 / 같은 모델을 세 환경에서 평가
experiment.train_joint()


In [ ]:
# 08 · 같은 선택 모델 Easy 테스트
_ = experiment.test("easy")


In [ ]:
# 09 · 같은 선택 모델 Medium 테스트
_ = experiment.test("medium")


In [ ]:
# 10 · 같은 선택 모델 Hard 테스트
_ = experiment.test("hard")


In [ ]:
# 11 · 공동 학습 결과
_ = experiment.report()


In [ ]:
# 12 · 선택 사항 / 같은 모델로 각 난이도 최종 100회
experiment.final_evaluation()


In [ ]:
# 13 · 동일 체크포인트 하나로 세 난이도 패키지
_ = experiment.package()
